# Fine-Tune Qwen2.5-7B for Arabic Teaching (SageMaker)

**Model:** Qwen2.5-7B-Instruct with LoRA (using Unsloth)

**Training data:** 99 conversations with keyword-based routing

**Method:** LoRA (rank=32, alpha=64)

**Max sequence length:** 2048 tokens

**Output:** 4-bit quantized model (~2GB)

---

## Setup for SageMaker

1. **Instance Type:** ml.g4dn.xlarge or ml.g5.xlarge (T4/A10G GPU)
2. **HF Token:** Set as environment variable `HF_TOKEN` or in cell below
3. **Training data:** Upload `agent1_teaching_training_data.jsonl` to EFS at `/home/sagemaker-user/user-default-efs/data/`
4. **Run all cells** (~30-40 minutes)
5. **Model output:** Saved to EFS at `/home/sagemaker-user/user-default-efs/models/qwen-7b-arabic-teaching/`

## Cell 1: Install Dependencies

In [ ]:
%%capture
# Simple installation - use SageMaker's default PyTorch
!pip install --upgrade torchvision
!pip install unsloth trl transformers datasets accelerate bitsandbytes
!pip install -U peft
!pip uninstall -y pyarrow
!pip install pyarrow --no-cache-dir

## ⚠️ RESTART KERNEL - Then run all cells below

## Cell 2: HuggingFace Auth

In [ ]:
# Check what's using space
!du -sh /home/sagemaker-user/* 2>/dev/null | sort -h

# Clear everything possible from default storage
!rm -rf ~/.cache/*
!rm -rf /home/sagemaker-user/.cache/*


import os

from huggingface_hub import login

EFS_ROOT = "/home/sagemaker-user/user-default-efs"
HF_CACHE = os.path.join(EFS_ROOT, "hf_cache")

# 1. Create the directories on the BIG drive
os.makedirs(HF_CACHE, exist_ok=True)

# 2. Redirect ALL possible cache/temp variables
os.environ["HF_HOME"] = HF_CACHE
os.environ["TRANSFORMERS_CACHE"] = os.path.join(EFS_ROOT, "hf_cache")
os.environ["HF_DATASETS_CACHE"] = os.path.join(EFS_ROOT, "hf_cache")

# Now check space
!df -h /home/sagemaker-user

print(f"✅ All traffic redirected to EFS: {HF_CACHE}")
login(token="<HF_TOKEN>")

## Cell 3: Imports & Check GPU

In [ ]:
import gc
import json
import os

import torch

# IMPORTANT: Disable Unsloth statistics BEFORE importing Unsloth
os.environ["UNSLOTH_DISABLE_LOG_STATS"] = "1"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

from datasets import Dataset
from trl import SFTTrainer
from unsloth import FastLanguageModel

print(f"PyTorch: {torch.__version__}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")
if torch.cuda.is_available():
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️  No GPU detected! Training will be very slow.")

## Cell 4: Configuration

In [ ]:
# EFS paths (adjust if your EFS mount is different)
EFS_ROOT = "/home/sagemaker-user/user-default-efs"
DATA_DIR = f"{EFS_ROOT}/data"

# Training data path
DATA_PATH = f"{DATA_DIR}/agent1_teaching_training_data.jsonl"
OUTPUT_DIR = f"{EFS_ROOT}/models/qwen-7b-arabic-teaching"

# Model config
MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"
MAX_SEQ_LENGTH = 2048
LOAD_IN_4BIT = True

# LoRA config
LORA_R = 32
LORA_ALPHA = 64
LORA_DROPOUT = 0.1
LORA_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj"]

# Training config
BATCH_SIZE = 2
GRAD_ACCUM = 4
EPOCHS = 5
LR = 2e-4
WARMUP = 0.03

print("=" * 60)
print("🚀 Fine-Tuning Qwen2.5-7B for Arabic Teaching")
print("=" * 60)
print(f"Model: {MODEL_NAME}")
print(f"Data: {DATA_PATH}")
print(f"Output: {OUTPUT_DIR}")
print(f"Batch: {BATCH_SIZE}, Accum: {GRAD_ACCUM}, Epochs: {EPOCHS}")
print(f"Max Seq Length: {MAX_SEQ_LENGTH}")
print("=" * 60)

# Verify data file exists
if not os.path.exists(DATA_PATH):
    print(f"\n❌ Training data not found at: {DATA_PATH}")
    print("\nPlease upload agent1_teaching_training_data.jsonl to EFS at:")
    print(f"   {os.path.dirname(DATA_PATH)}/")
else:
    print("\n✅ Training data found")

# Verify EFS cache is setup
print(f"\n📁 HuggingFace cache: {os.environ.get('HF_HOME', 'NOT SET')}")
if os.path.exists(os.environ.get("HF_HOME", "")):
    print("✅ EFS cache directory exists")
else:
    print("❌ EFS cache directory NOT found - run Cell 2 first!")

## Cell 5: Load Training Data

In [ ]:
conversations = []
with open(DATA_PATH, encoding="utf-8") as f:
    for line in f:
        if line.strip():
            conversations.append(json.loads(line))

print(f"✅ Loaded {len(conversations)} conversations")
print(f"   Steps/epoch: {len(conversations) // (BATCH_SIZE * GRAD_ACCUM)}")
print(f"   Total steps: {(len(conversations) // (BATCH_SIZE * GRAD_ACCUM)) * EPOCHS}")

## Cell 6: Load Model with Unsloth

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=LOAD_IN_4BIT,
    token="<HF_TOKEN>",
)
print(f"✅ Model loaded: {model.num_parameters() / 1e9:.2f}B params")

## Cell 7: Format Dataset

In [ ]:
# Manually format and tokenize dataset with single-process to avoid pickle errors
def format_and_tokenize(example):
    # Format with chat template
    text = tokenizer.apply_chat_template(
        example["messages"], tokenize=False, add_generation_prompt=False
    )
    # Tokenize
    tokenized = tokenizer(
        text,
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
        padding=False,
    )
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized

dataset = Dataset.from_list(conversations)
# Use num_proc=None to force single-process mode (avoids pickle errors)
dataset = dataset.map(format_and_tokenize, remove_columns=["messages"], num_proc=None)
print(f"✅ Dataset tokenized: {len(dataset)} examples")

In [ ]:
# Validation split
eval_size = 15  # Use 15 examples for validation
train_dataset = dataset.select(range(len(dataset) - eval_size))
eval_dataset = dataset.select(range(len(dataset) - eval_size, len(dataset)))

print(f"✅ Train: {len(train_dataset)}, Eval: {len(eval_dataset)}")

## Cell 8: Setup Trainer

In [ ]:
import torch
from peft import LoraConfig
from trl import SFTConfig

# 1. Clear memory
torch.cuda.empty_cache()
gc.collect()

# 2. Define the PEFT configuration (matches your LORA settings)
peft_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=LORA_MODULES,
    bias="none",
    task_type="CAUSAL_LM",
)

model.add_adapter(peft_config, adapter_name="my_adapter")

# 3. Define the SFTConfig (dataset already tokenized)
sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_text_field=None,  # Already tokenized
    packing=False,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    num_train_epochs=EPOCHS,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    optim="adamw_8bit",
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="loss",
    logging_steps=1,
    weight_decay=0.01,
    lr_scheduler_type="linear",
    seed=42,
    report_to="none",
    average_tokens_across_devices=False,
    dataloader_num_workers=0,
)

# 4. Initialize Trainer (no processing_class since data is pre-tokenized)
trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    args=sft_config,
)

trainer.model.set_adapter("my_adapter")

print("✅ Trainer ready!")

## Cell 9: Train (~30-40 min on g4dn.xlarge)

In [ ]:
import logging

# Disable the specific logger that's crashing on the closed Jupyter stream
logging.getLogger("transformers.trainer").setLevel(logging.ERROR)

print("🚀 Training starting...")

try:
    trainer.train()
except ValueError as e:
    if "I/O operation on closed file" in str(e):
        print("Caught Jupyter logging error, but training is proceeding in the background.")
    else:
        raise e

print("\n✅ Training complete! Check your EFS 'models' folder for checkpoints!")

## Cell 10: Save Model to EFS

In [ ]:
import os
import tarfile

# Save to EFS (persists across notebook sessions)
lora_output_dir = f"{OUTPUT_DIR}/lora_adapters"
model.save_pretrained(lora_output_dir)
tokenizer.save_pretrained(lora_output_dir)

print(f"✅ Saved to EFS: {lora_output_dir}")
print("\n📁 Model persisted on EFS at:")
print(f"   {lora_output_dir}")

print("\n📦 Creating downloadable archive...")

# Create tarball
archive_path = f"{OUTPUT_DIR}/qwen-7b-arabic-teaching-lora.tar.gz"
with tarfile.open(archive_path, "w:gz") as tar:
    tar.add(lora_output_dir, arcname="lora_adapters")

archive_size = os.path.getsize(archive_path) / (1024 * 1024)
print(f"✅ Archive created: {archive_path} ({archive_size:.1f} MB)")

print("\n📥 To download:")
print("   1. File browser → archive path above")
print("   2. Right-click → Download")
print("   3. Extract on your machine")

print("\nModel files in archive:")
for file in os.listdir(lora_output_dir):
    file_path = os.path.join(lora_output_dir, file)
    size_mb = os.path.getsize(file_path) / (1024 * 1024)
    print(f"   - {file} ({size_mb:.1f} MB)")

---

# Evaluation Tests

**Note:** Model is already loaded in memory from training above.

## Cell 11: Load Evaluation Tests

In [ ]:
import json

# Load evaluation tests
EVAL_PATH = f"{EFS_ROOT}/tests/sagemaker_evaluation_tests.jsonl"

eval_tests = []
with open(EVAL_PATH, encoding="utf-8") as f:
    for line in f:
        if line.strip():
            eval_tests.append(json.loads(line))

print(f"✅ Loaded {len(eval_tests)} evaluation tests")
print(f"\nTest scenarios: {len(set(t['scenario'].split('_')[0] + '_' + t['scenario'].split('_')[1] for t in eval_tests))}")
print(f"Tests per scenario: 2")

## Cell 12: Run Keyword Detection Tests

In [ ]:
print("=" * 80)
print("RUNNING KEYWORD DETECTION TESTS")
print("=" * 80)
print()

results = []
passed = 0
failed = 0

for test in eval_tests:
    scenario = test["scenario"]
    user_input = test["user_input"]
    expected_keyword = test["expected_keyword"]
    messages = test["messages"]
    
    # Format messages for model
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    
    # Generate response
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    outputs = model.generate(
        **inputs,
        max_new_tokens=256,
        temperature=0.7,
        do_sample=True,
    )
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    # Extract only the assistant's response (after the last "assistant" marker)
    if "assistant" in response:
        response = response.split("assistant")[-1].strip()
    
    # Check if expected keyword is in response (case-insensitive)
    keyword_found = expected_keyword.lower() in response.lower()
    
    result = {
        "scenario": scenario,
        "user_input": user_input,
        "expected_keyword": expected_keyword,
        "response": response,
        "keyword_found": keyword_found,
        "passed": keyword_found
    }
    
    results.append(result)
    
    if keyword_found:
        passed += 1
        status = "✅ PASS"
    else:
        failed += 1
        status = "❌ FAIL"
    
    print(f"{status} - {scenario}")
    print(f"   User: {user_input}")
    print(f"   Expected: '{expected_keyword}'")
    print(f"   Response: {response[:100]}...")
    print()

print("=" * 80)
print("RESULTS")
print("=" * 80)
print(f"Total tests: {len(eval_tests)}")
print(f"Passed: {passed} ({passed/len(eval_tests)*100:.1f}%)")
print(f"Failed: {failed} ({failed/len(eval_tests)*100:.1f}%)")
print("=" * 80)

## Cell 13: Save Evaluation Results

In [ ]:
from datetime import datetime

# Prepare results
output = {
    "metadata": {
        "model": "Qwen2.5-7B Teaching (LoRA) - Keyword Detection",
        "evaluation_date": datetime.now().isoformat(),
        "device": "SageMaker GPU",
        "test_type": "Keyword detection for orchestrator routing",
    },
    "summary": {
        "total_tests": len(eval_tests),
        "passed": passed,
        "failed": failed,
        "pass_rate": passed / len(eval_tests),
    },
    "results": results,
}

# Save results
output_path = f"{OUTPUT_DIR}/keyword_detection_results.json"
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(output, f, indent=2, ensure_ascii=False)

print(f"✅ Results saved to {output_path}")
print(f"\n📥 Download this file to analyze results locally")

## Cell 14: Show Failed Tests

In [ ]:
# Show failed tests for debugging
failed_tests = [r for r in results if not r["passed"]]

if failed_tests:
    print("=" * 80)
    print(f"FAILED TESTS ({len(failed_tests)})")
    print("=" * 80)
    print()
    
    for result in failed_tests:
        print(f"Scenario: {result['scenario']}")
        print(f"User Input: {result['user_input']}")
        print(f"Expected Keyword: '{result['expected_keyword']}'")
        print(f"Model Response:")
        print(f"   {result['response']}")
        print()
        print("-" * 80)
        print()
else:
    print("🎉 All tests passed!")

## Cell 15: Test Numeric/Short Responses